In [1]:
import os
from pathlib import Path

import pandas as pd
from databricks import sql
from dotenv import load_dotenv

load_dotenv()

# folder of this file; in a notebook there is no __file__, so use the working directory
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

INPUT_DIR = BASE_DIR / "input"
INPUT_DIR.mkdir(exist_ok=True)

QUERY = """
    SELECT DISTINCT code, description
    FROM synthea.1_bronze.conditions
    WHERE LOWER(description) NOT LIKE '%acute%'
        AND LOWER(description) NOT LIKE '%chronic%'
        AND LOWER(description) NOT LIKE '%disorder%'
    ORDER BY code
"""

with sql.connect(
    server_hostname=os.getenv("DATABRICKS_HOST"),
    http_path=os.getenv("DATABRICKS_HTTP_PATH"),
    access_token=os.getenv("DATABRICKS_TOKEN"),
) as conn:
    with conn.cursor() as cur:
        cur.execute(QUERY)
        df = pd.DataFrame([r.asDict() for r in cur.fetchall()])

df["code"] = df["code"].astype(str)
df.to_csv(INPUT_DIR / "conditions_to_classify.csv", index=False)
print(f"Wrote {len(df)} rows to {INPUT_DIR / 'conditions_to_classify.csv'}")

[WARN] pyarrow is not installed by default since databricks-sql-connector 4.0.0,any arrow specific api (e.g. fetchmany_arrow) and cloud fetch will be disabled.If you need these features, please run pip install pyarrow or pip install databricks-sql-connector[pyarrow] to install


Wrote 139 rows to c:\repos\healthcare-analytics-engineering\scripts\classify_conditions\input\conditions_to_classify.csv
